# Tutorial 05 — Adaptive Intelligence

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eddmpython/vectrix/blob/master/notebooks/tutorials/05_adaptive.ipynb)

**These features are unique to Vectrix** — not available in statsforecast, Prophet, Darts, or any other library. Adaptive intelligence means your forecasts respond to changing conditions: regime shifts, self-correction as new data arrives, business constraints, and DNA-driven model selection.

| What you'll learn | Time |
|:------------------|:-----|
| Regime detection | 2 min |
| Forecast DNA | 2 min |
| Self-healing forecast | 2 min |
| Constraint-aware forecasting | 2 min |

In [ ]:
!pip install -q vectrix

## 1. Regime Detection

Real-world data rarely follows a single pattern. Markets alternate between bull and bear phases, retail demand shifts between peak and off-season. Vectrix detects these **regimes** automatically.

In [ ]:
from vectrix import RegimeDetector
import numpy as np

np.random.seed(42)
regime1 = np.random.randn(50) * 2 + 100
regime2 = np.random.randn(50) * 5 + 130
regime3 = np.random.randn(50) * 2 + 110
data = np.concatenate([regime1, regime2, regime3])

detector = RegimeDetector(nRegimes=3)
result = detector.detect(data)

print(f"Current regime: {result.currentRegime}")
print(f"Number of regimes: {len(result.regimeStats)}")
for label, stats in result.regimeStats.items():
    print(f"  Regime {label}: mean={stats['mean']:.2f}, std={stats['std']:.2f}")

### RegimeDetector Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `nRegimes` | `2` | Number of regimes to detect |

### RegimeResult Attributes

| Attribute | Type | Description |
|-----------|------|-------------|
| `currentRegime` | `int` | Index of the current (most recent) regime |
| `regimeStats` | `dict` | Per-regime statistics (mean, std, size) |
| `regimeSequence` | `np.ndarray` | Regime label for each observation |

## 2. Regime-Aware Forecasting

Identifies which regime the series is currently in and uses the model that performed best during similar past regimes.

In [ ]:
from vectrix import RegimeAwareForecaster
import numpy as np

np.random.seed(42)
data = np.random.randn(200).cumsum() + 100

raf = RegimeAwareForecaster()
result = raf.forecast(data, steps=30, period=7)

print(f"Current regime:    {result.currentRegime}")
print(f"Model per regime:  {result.modelPerRegime}")
print(f"Predictions (5):   {result.predictions[:5].round(2)}")

## 3. Forecast DNA

Every time series has a unique statistical signature. DNA profiling extracts 65+ features to create a deterministic fingerprint.

In [ ]:
from vectrix import ForecastDNA
import numpy as np

dna = ForecastDNA()

t = np.arange(100)
data = 50 + 0.3 * t + 15 * np.sin(2 * np.pi * t / 12) + np.random.randn(100) * 3

profile = dna.analyze(data, period=12)

print(f"Fingerprint:  {profile.fingerprint}")
print(f"Difficulty:   {profile.difficulty} ({profile.difficultyScore:.0f}/100)")
print(f"Category:     {profile.category}")
print(f"Recommended:  {profile.recommendedModels[:5]}")

In [ ]:
# Using DNA for decision making
from vectrix import ForecastDNA, forecast

dna = ForecastDNA()
profile = dna.analyze(data, period=12)

if profile.difficultyScore > 70:
    print("Hard series — consider using ensemble or longer history.")
elif profile.category == "intermittent":
    print("Sparse demand — Croston variants recommended.")
else:
    result = forecast(data, steps=12)
    print(f"Forecast with {result.model}, MAPE={result.mape:.2f}%")

## 4. Self-Healing Forecast

Forecasts degrade over time. **Self-healing** monitors errors as actual values arrive and automatically adjusts remaining predictions.

In [ ]:
from vectrix import SelfHealingForecast, forecast
import numpy as np

np.random.seed(42)
data = np.random.randn(100).cumsum() + 200
result = forecast(data, steps=14)

healer = SelfHealingForecast(
    result.predictions,
    result.lower,
    result.upper,
    data,
)

# Simulate actual values arriving
actual_day1_to_5 = np.array([198.5, 201.2, 195.8, 203.4, 199.1])
healer.observe(actual_day1_to_5)

report = healer.getReport()
print(f"Health:      {report.overallHealth} ({report.healthScore:.1f}/100)")
print(f"Observed:    {report.totalObserved}")
print(f"Corrected:   {report.totalCorrected}")
print(f"Improvement: {report.improvementPct:.1f}%")

In [ ]:
# Get updated forecast with corrections
updated_preds, updated_lower, updated_upper = healer.getUpdatedForecast()
print(f"Original remaining: {result.predictions[5:].round(1)}")
print(f"Updated remaining:  {updated_preds[5:].round(1)}")

## 5. Constraint-Aware Forecasting

Predictions can go negative (impossible for unit sales), exceed warehouse capacity, or show unrealistic swings. Constraints enforce domain knowledge.

In [ ]:
from vectrix import ConstraintAwareForecaster, Constraint, forecast
import numpy as np

np.random.seed(42)
data = np.random.randn(100).cumsum() + 500
result = forecast(data, steps=14)

caf = ConstraintAwareForecaster()
constrained = caf.apply(
    result.predictions,
    result.lower,
    result.upper,
    constraints=[
        Constraint('non_negative', {}),
        Constraint('range', {'min': 100, 'max': 5000}),
        Constraint('capacity', {'capacity': 3000}),
    ]
)

print(f"Original:    {result.predictions[:5].round(1)}")
print(f"Constrained: {constrained.predictions[:5].round(1)}")

### All Constraint Types

| Type | Parameters | Description |
|------|-----------|-------------|
| `non_negative` | `{}` | Ensures all predictions >= 0 |
| `range` | `{'min': N, 'max': M}` | Clips to [min, max] |
| `capacity` | `{'capacity': N}` | Caps at ceiling |
| `yoy_change` | `{'maxPct': N, 'historicalData': arr}` | Limits year-over-year change |
| `sum_constraint` | `{'window': N, 'maxSum': M}` | Limits sum within rolling windows |
| `monotone` | `{'direction': 'increasing'}` | Forces monotonic direction |
| `ratio` | `{'minRatio': N, 'maxRatio': M}` | Limits consecutive value ratio |
| `custom` | `{'fn': callable}` | Custom constraint function |

In [ ]:
# Custom constraint example: weekend boost
import numpy as np

def weekend_boost(predictions, lower, upper):
    boosted = predictions.copy()
    lo = lower.copy()
    hi = upper.copy()
    for i in range(len(boosted)):
        if i % 7 in [5, 6]:
            boosted[i] *= 1.2
            lo[i] *= 1.2
            hi[i] *= 1.2
    return boosted, lo, hi

custom_result = caf.apply(
    result.predictions, result.lower, result.upper,
    constraints=[Constraint('custom', {'fn': weekend_boost})]
)
print(f"With weekend boost: {custom_result.predictions[:7].round(1)}")

## 6. Complete Example

Full adaptive workflow: profile, detect regimes, forecast, apply constraints.

In [ ]:
import numpy as np
from vectrix import (
    forecast, ForecastDNA, RegimeDetector,
    SelfHealingForecast, ConstraintAwareForecaster, Constraint
)

np.random.seed(42)
data = np.random.randn(200).cumsum() + 500

# Step 1: DNA profiling
dna = ForecastDNA()
profile = dna.analyze(data, period=7)
print(f"Difficulty:  {profile.difficulty}")
print(f"Recommended: {profile.recommendedModels[:3]}")

# Step 2: Regime detection
detector = RegimeDetector(nRegimes=2)
regimes = detector.detect(data)
print(f"Current regime: {regimes.currentRegime}")

# Step 3: Forecast
result = forecast(data, steps=14)
print(f"Model: {result.model}, MAPE: {result.mape:.2f}%")

# Step 4: Apply constraints
caf = ConstraintAwareForecaster()
constrained = caf.apply(
    result.predictions, result.lower, result.upper,
    constraints=[
        Constraint('non_negative', {}),
        Constraint('range', {'min': 300, 'max': 800}),
    ]
)
print(f"Constrained: {constrained.predictions[:5].round(1)}")

---

**Next:** [Tutorial 06 — Business Intelligence](06_business.ipynb)